# 3.7 · 文本特征工程 / Text Feature Engineering

> **课程定位 / Where this fits**
> **Part 3 第 7 课**。数值/类别/时间讲完, 进入**文本**。如何把一句话变成模型能吃的数字向量？本课走经典路线：清洗 → 词袋 → TF-IDF → n-gram。**这是 Part 11 经典 NLP 的预热**, 也是至今仍打硬仗的强基线。
> Turning sentences into vectors: cleaning -> bag-of-words -> TF-IDF -> n-grams. The warm-up to Part 11, and a baseline that still competes.

> 💡 **面试相关 / Interview-relevant**
> - "TF-IDF 的公式和直觉" ★★★★★（NLP 必考）
> - "词袋模型丢了什么信息" ★★★★（词序）
> - "n-gram 解决什么" ★★★
> - "停用词 / 词干化要不要做" ★★★

---

## 学习目标 / Learning Objectives
1. 理解文本向量化的核心难题：**变长 → 定长、稀疏、词序丢失**。
2. 掌握 **CountVectorizer (词袋) → TfidfVectorizer** 的演进与公式。
3. 用 **n-gram** 找回部分词序信息。
4. 在 SMS 垃圾短信上**端到端**做一个文本分类器（接 0.9 节朴素贝叶斯）。
5. 知道现代 embedding 路线（Part 9/12）何时取代 TF-IDF。

## 目录 / TOC
1. [文本向量化的三大难题 ⭐](#1)
2. [📱 数据：SMS 垃圾短信](#2)
3. [文本清洗](#3)
4. [词袋 CountVectorizer](#4)
5. [TF-IDF ⭐](#5)
6. [n-gram：找回词序](#6)
7. [端到端：垃圾短信分类](#7)
8. [TF-IDF vs Embedding 何时换](#8)
9. [小结](#9)


<a id="1"></a>
## 1. 文本向量化的三大难题 ⭐ / Three Challenges

模型要定长数值向量, 但文本是**变长的字符序列**。核心难题：

| 难题 | 说明 | 经典方案的应对 |
|---|---|---|
| **变长 → 定长** | "hi" 和一篇文章长度差千倍 | 词袋: 向量维度 = 词表大小, 与文本长度无关 |
| **高维稀疏** | 词表几万, 单条文本只含几十词 | 稀疏矩阵存储 (只存非零) |
| **词序丢失** ⚠ | "狗咬人" vs "人咬狗" 词袋完全相同 | n-gram 部分弥补; embedding/Transformer 才真正解决 (Part 12) |

**经典路线的根本假设**：**词袋 (Bag-of-Words)** —— 把文本看成"一袋无序的词", 只数词频, 不管顺序。粗糙但惊人地有效（垃圾邮件、情感分类至今强基线）。
The bag-of-words assumption — treat text as an unordered bag of words. Crude but a remarkably strong baseline.


<a id="2"></a>
## 2. 📱 数据：SMS 垃圾短信 / SMS Spam

复用 0.9 节朴素贝叶斯的 mini SMS 数据集（扩充一点）。0.9 我们手写了 NB; 这次专注**特征侧**——怎么把短信变成向量。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")

# mini SMS 数据集 (扩充版) / expanded mini SMS dataset
spam = [
    "WIN a free iPhone today click here now","URGENT you won a 1000 dollar prize call now",
    "Free entry weekly prize draw text WIN to 12345","Congratulations selected for free vacation",
    "Click link to claim your free reward now","Limited offer act now claim free gift card",
    "You have won cash prize call this number","Free ringtones text the number to win",
    "Claim your free holiday now urgent reply","Win big money click the link below now",
    "Exclusive offer free trial click now","Your account won a prize claim immediately",
]
ham = [
    "Hey are we still meeting for lunch tomorrow","Can you pick up some milk on your way home",
    "Happy birthday hope you have a great day","I will be late to the meeting sorry",
    "Did you watch the game last night","Lets grab coffee this weekend if free",
    "Mom called she wants you to call back","Running errands will see you at home tonight",
    "Thanks for the help yesterday really appreciate it","What time should I come over for dinner",
    "The report is due friday can we discuss","See you at the gym after work today",
]
df = pd.DataFrame({"text": spam+ham, "label": ["spam"]*len(spam)+["ham"]*len(ham)})
df = df.sample(frac=1, random_state=0).reset_index(drop=True)
print(f"{len(df)} 条短信, {(df.label=='spam').sum()} spam / {(df.label=='ham').sum()} ham")
df.head(3)


<a id="3"></a>
## 3. 文本清洗 / Text Cleaning

向量化前的标准预处理（各步骤是否做取决于任务）：

| 步骤 | 作用 | 注意 |
|---|---|---|
| 小写化 | "Free"="free" | 几乎总做 |
| 去标点/数字 | 减噪 | 但 "$1000" 对垃圾短信是信号, 别盲目去 |
| 分词 (tokenize) | 切成词 | 中文需专门分词器 (jieba) |
| 去停用词 | 删 the/a/is | TF-IDF 会自动降权, 不一定要删 |
| 词干化/词形还原 | running→run | 减小词表; lemmatize 比 stem 准 |


In [ ]:
import re

def clean(text):
    text = text.lower()                          # 小写
    text = re.sub(r"[^a-z\s]", " ", text)        # 只留字母 (演示用; 真实任务可能保留数字/$)
    return re.sub(r"\s+", " ", text).strip()

df["clean"] = df["text"].apply(clean)
print("清洗前后对比:")
for i in [0, 1]:
    print(f"  原始: {df.text.iloc[i]}")
    print(f"  清洗: {df.clean.iloc[i]}\n")


<a id="4"></a>
## 4. 词袋 CountVectorizer / Bag-of-Words

**最朴素的向量化**：建词表 → 每条文本变成"各词出现次数"的向量。


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()
X_count = cv.fit_transform(df["clean"])     # 稀疏矩阵 / sparse matrix
vocab = cv.get_feature_names_out()
print(f"词表大小: {len(vocab)}")
print(f"特征矩阵: {X_count.shape} (稀疏, 只存非零)")
print(f"前 15 个词: {list(vocab[:15])}")

# 看一条短信的词袋向量 (非零部分) / one message's BoW vector
i = 0
row = X_count[i].toarray().ravel()
present = {vocab[j]: row[j] for j in np.nonzero(row)[0]}
print(f"\n短信: '{df.clean.iloc[i]}'")
print(f"词袋 (非零): {present}")


**词袋的问题**：常见词（"the", "to"）频次高但**没区分力**——它在 spam 和 ham 里都常见。需要一种机制**降低无区分力词的权重**——这就是 TF-IDF。
Common words have high counts but no discriminative power — TF-IDF fixes this by down-weighting them.


<a id="5"></a>
## 5. TF-IDF ⭐ / Term Frequency–Inverse Document Frequency

**核心思想**：一个词的重要性 = **在本文档里有多常见 (TF) × 在所有文档里有多罕见 (IDF)**。

$$\text{TF-IDF}(t, d) = \underbrace{\text{tf}(t, d)}_{\text{词 t 在文档 d 的频率}} \times \underbrace{\log\frac{N}{\text{df}(t)}}_{\text{IDF: 含 t 的文档越少, 权重越高}}$$

- **TF 高**：在本文档里反复出现 → 对本文档重要
- **IDF 高**：只在少数文档出现 → 有区分力（"free"只在垃圾短信里多 → 高 IDF 权重）
- **常见词**（the/to, 几乎每篇都有）→ IDF ≈ 0 → 被自动压低

> 💡 sklearn 的 IDF 默认是 $\log\frac{N+1}{\text{df}+1}+1$（平滑版, 防止除零和负权重）。


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(df["clean"])

# 看哪些词 IDF 最高(最有区分力) / highest-IDF (most discriminative) words
idf = pd.Series(tfidf.idf_, index=tfidf.get_feature_names_out()).sort_values(ascending=False)
print("IDF 最高的词 (最罕见/最有区分力):")
print(idf.head(8).round(2).to_dict())
print("\nIDF 最低的词 (最常见/最没区分力):")
print(idf.tail(8).round(2).to_dict())
print("\n→ 'free','win','urgent' 等垃圾词 IDF 高; 'you','to' 等常见词 IDF 低 — 符合直觉")


In [ ]:
# 哪些词最能区分 spam vs ham? (用 TF-IDF + 类别均值差) / discriminative words
import numpy as np
spam_mask = (df.label == "spam").values
vocab = tfidf.get_feature_names_out()
spam_mean = np.asarray(X_tfidf[spam_mask].mean(axis=0)).ravel()
ham_mean  = np.asarray(X_tfidf[~spam_mask].mean(axis=0)).ravel()
diff = pd.Series(spam_mean - ham_mean, index=vocab).sort_values()

print("最 'ham' 的词:", list(diff.head(5).index))
print("最 'spam' 的词:", list(diff.tail(5).index))


<a id="6"></a>
## 6. n-gram：找回词序 / n-grams

词袋丢了词序（"not good" 和 "good not" 一样）。**n-gram = 连续 n 个词作为一个特征**, 部分找回局部词序。

- **unigram** (1-gram): `["click", "now"]`
- **bigram** (2-gram): `["click now"]` ← "click now" 整体是垃圾信号
- **trigram**: `["claim free prize"]`


In [ ]:
# unigram vs unigram+bigram / compare
cv1 = CountVectorizer(ngram_range=(1,1))
cv2 = CountVectorizer(ngram_range=(1,2))     # unigram + bigram
X1, X2 = cv1.fit_transform(df.clean), cv2.fit_transform(df.clean)
print(f"unigram 词表: {X1.shape[1]} 个特征")
print(f"unigram+bigram: {X2.shape[1]} 个特征 (维度增加, 但抓住了'click now'等短语)")

# 看 bigram 特征 / show bigram features
bigrams = [w for w in cv2.get_feature_names_out() if " " in w]
print(f"\n部分 bigram: {bigrams[:8]}")
print("\n⚠ n 越大特征越多越稀疏 → 通常止于 bigram/trigram; 配合 min_df 过滤罕见 n-gram")


<a id="7"></a>
## 7. 端到端：垃圾短信分类 / End-to-end Spam Classifier

把文本特征接上分类器（用 Pipeline 防泄漏, 3.12 正题）。对照 0.9 节手写的朴素贝叶斯——这次用 sklearn 全家桶。


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

X, y = df["clean"], (df.label == "spam").astype(int)

# 三个 pipeline: 词袋+NB / TF-IDF+NB / TF-IDF+逻辑回归 / three pipelines
pipes = {
    "BoW + NaiveBayes":       Pipeline([("vec", CountVectorizer()), ("clf", MultinomialNB())]),
    "TF-IDF + NaiveBayes":    Pipeline([("vec", TfidfVectorizer()), ("clf", MultinomialNB())]),
    "TF-IDF(1,2) + LogReg":   Pipeline([("vec", TfidfVectorizer(ngram_range=(1,2))),
                                        ("clf", LogisticRegression(max_iter=1000))]),
}
for name, pipe in pipes.items():
    acc = cross_val_score(pipe, X, y, cv=4).mean()
    print(f"{name:<26}: CV 准确率 = {acc:.1%}")
print("\n💡 Pipeline 保证 vectorizer 只在每折的 train 上 fit — 自动防泄漏")


In [ ]:
# 训练最终模型 + 看学到的"垃圾词" / final model + learned spam words
pipe = pipes["TF-IDF(1,2) + LogReg"].fit(X, y)
vec, clf = pipe.named_steps["vec"], pipe.named_steps["clf"]
coef = pd.Series(clf.coef_.ravel(), index=vec.get_feature_names_out())
print("逻辑回归学到的最强垃圾信号词 (系数最大):")
print(coef.sort_values(ascending=False).head(6).round(2).to_dict())

# 预测新短信 / predict new messages
new = ["free prize click now to win", "hey want to grab dinner tonight"]
print(f"\n新短信预测:")
for txt, p in zip(new, pipe.predict([clean(t) for t in new])):
    print(f"  [{'spam' if p else 'ham'}] {txt}")


<a id="8"></a>
## 8. TF-IDF vs Embedding 何时换 / When to Switch

| | TF-IDF (经典) | Embedding (现代, Part 9/12) |
|---|---|---|
| 表示 | 高维稀疏 (词表大小) | 低维稠密 (如 768 维) |
| 词序 | 丢失 (n-gram 部分弥补) | Transformer 完整建模 |
| 语义 | 无 ("good"/"great" 是不同列) | 有 (近义词向量相近) |
| 数据需求 | 少 (小数据强基线) | 多 (或用预训练) |
| 速度 | 极快 | 慢 (需 GPU) |
| 何时用 | **小数据 / 强基线 / 可解释 / 关键词主导任务** ⭐ | 语义/上下文重要的任务 |

**实战智慧**：**永远先跑 TF-IDF + 逻辑回归做基线**——它快、可解释、常常已经够好。垃圾邮件、文档分类、关键词检测里 TF-IDF 至今不输大模型且便宜千倍。
Always run TF-IDF + logistic regression as a baseline first — fast, interpretable, often good enough. Spam and keyword-driven tasks still favor it.


<a id="9"></a>
## 9. 小结 / Summary

```
文本三难题: 变长→定长 / 高维稀疏 / 词序丢失
向量化演进:
  CountVectorizer (词袋, 数词频)
    → TF-IDF (× IDF 压低常见词, 抬高有区分力词) ⭐
    → + n-gram (bigram 抓'click now'等短语, 找回部分词序)
端到端: Pipeline(TfidfVectorizer → 分类器), 自动防泄漏
现代: embedding (语义 + 词序, Part 12); 但 TF-IDF 仍是强基线
```

### 💡 面试速查
1. **TF-IDF 公式**: tf(t,d) × log(N/df(t)); 直觉 = 本文常见 × 全局罕见
2. **词袋丢词序** → n-gram 部分弥补 → Transformer 真正解决
3. **常见词被 IDF 自动压低** (the/to 权重≈0)
4. **永远先跑 TF-IDF + LogReg 基线**
5. **Pipeline 防 vectorizer 泄漏**(只 fit train)

### 下一节
**3.8 图像特征基础**——文本讲完, 最后一个数据类型：图像。像素 / HOG / 颜色直方图（深度 CNN 前的经典特征）。
